# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [16]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [17]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['PAISMBJIYQ', 'DRFJOMZYXH'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[16,  1,  9, 19, 13,  2, 10,  9, 25, 17],
       [ 4, 18,  6, 10, 15, 13, 26, 25, 24,  8]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 17, 25,  9, 10,  2, 13, 19,  9,  1],
       [ 0,  8, 24, 25, 26, 13, 15, 10,  6, 18]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[17, 25,  9, 10,  2, 13, 19,  9,  1, 16],
       [ 8, 24, 25, 26, 13, 15, 10,  6, 18,  4]], dtype=int32)>)


# 建立sequence to sequence 模型

In [22]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz = 27
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        self.encoder = tf.keras.layers.RNN(
            self.encoder_cell, return_sequences=True, return_state=True
        )
        self.decoder = tf.keras.layers.RNN(
            self.decoder_cell, return_sequences=True, return_state=True
        )
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        '''
        完成sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        '''
        enc_emb = self.embed_layer(enc_ids)
        _, enc_state = self.encoder(enc_emb)
        
        dec_emb = self.embed_layer(dec_ids)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state)
        logits = self.dense(dec_out)
        return logits
    
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)
        _, enc_state = self.encoder(enc_emb)
        return enc_state
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        inp_emb = self.embed_layer(x)
        state_list = [state] if not isinstance(state, (list, tuple)) else state
        h, new_state_list = self.decoder_cell(inp_emb, state_list)
        logits = self.dense(h)
        out = tf.argmax(logits, axis=-1)
        return out, new_state_list[0]

# Loss函数以及训练逻辑

In [19]:
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
        logits=logits, labels=labels
    )
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    grads = tape.gradient(loss, model.trainable_variables)
    grads_and_vars = [
        (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
    ]
    if not grads_and_vars:
        raise RuntimeError("梯度为空，请检查前向图是否可导。")
    optimizer.apply_gradients(grads_and_vars)
    return loss

def train(model, optimizer, seqlen, steps=1000):
    loss = 0.0
    for step in range(steps):
        _, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 200 == 0:
            print('step', step, ': loss', float(loss.numpy()))
    return loss

# 训练迭代

In [23]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3081729412078857
step 200 : loss 2.51375412940979
step 400 : loss 1.872389554977417
step 600 : loss 1.4443457126617432
step 800 : loss 1.1490176916122437


<tf.Tensor: shape=(), dtype=float32, numpy=0.9826569557189941>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [25]:
def sequence_reversal():
    def decode(init_state, steps=10):
        if isinstance(init_state, (list, tuple)):
            b_sz = tf.shape(init_state[0])[0]
            state = init_state[0]
        else:
            b_sz = tf.shape(init_state)[0]
            state = init_state
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[False, True, False, False, False, True, True, False, True, True, True, True, False, True, True, False, False, False, False, False, False, True, False, True, True, False, False, True, False, False, True, False]
[('ZFWUKWAJGC', 'CGJAWKUWFZ'), ('QJEWSGDYMB', 'BSBTWSWLSQ'), ('ONGMFHGDWK', 'KWDGHFMGNO'), ('KFTGCDUVYQ', 'QYVUDCGTFK'), ('YVTPXIJZXB', 'BXYJIXPTVY'), ('TSXSFCIAMY', 'XAAVCFSYSD'), ('DATCIFKXOG', 'OOXEFCCTAD'), ('BTSPPYYXAM', 'PAXUJGNAGU'), ('DUGFIFTSLP', 'PLSTFIFGUD'), ('OBMLATEMCI', 'ICMETALMBO'), ('WTOUMHPYPL', 'LPYPHMUOTW'), ('YRZPPCMDRF', 'GRCMCPSZRY'), ('PGLUIGVOXN', 'NXOVGIULGP'), ('VGBACIWFWY', 'UGHQGRCXGV'), ('NZSWVXASEO', 'ZJKRQMQSZW'), ('GRHZNUFIMK', 'EMPSYOZHRG'), ('SYLKHMYNOA', 'ANOZMHKLYS'), ('EKJYWFBCVJ', 'JVCBFWYJKE'), ('RXFAPQXZKG', 'SKGRWOAFXC'), ('TVSAHERRPG', 'GPRREHASVT'), ('XSNACKLRIW', 'WIRLKCANSX'), ('IBSGOAKUAT', 'TAUKAOGSBI'), ('WXUIDEJNUN', 'BUWMECIUXW'), ('OJGFNXTTXA', 'AXTTXNFGJO'), ('SVJTQAJUQI', 'IQUJAQTJVS'), ('GGYWQLFHFU', 'UFHFLQWYGG'), ('WCDHZN